In [1]:
!python --version

Python 3.10.12


In [2]:
# Local imports
from env.PushTEnv import PushTEnv

from models.diffusion_model import create_diffusion_model

from model_pipeline.PushTDataset import PushTStateDataset
from model_pipeline.training import train_diffusion_model
from model_pipeline.inference import run_inference_diffusion_model

pygame 2.1.2 (SDL 2.0.16, Python 3.10.12)
Hello from the pygame community. https://www.pygame.org/contribute.html


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/home/mya/Desktop/github/diffusion_exploration/venv-diff/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#@markdown ### **Imports**
# diffusion policy import
import os
import torch

# from typing import Tuple, Sequence, Dict, Union, Optional
import numpy as np

from skvideo.io import vwrite
# from IPython.display import Video
import gdown


In [4]:
from huggingface_hub.utils import IGNORE_GIT_FOLDER_PATTERNS
#@markdown ### **Env Demo**
#@markdown Standard Gym Env (0.21.0 API)

# 0. create env object
env = PushTEnv()

# 1. seed env for initial state.
# Seed 0-200 are used for the demonstration dataset.
env.seed(1000)

# 2. must reset before use
obs, IGNORE_GIT_FOLDER_PATTERNS = env.reset()

# 3. 2D positional action space [0,512]
action = env.action_space.sample()

# 4. Standard gym step method
obs, reward, terminated, truncated, info = env.step(action)

# prints and explains each dimension of the observation and action vectors
with np.printoptions(precision=4, suppress=True, threshold=5):
    print("Obs: ", repr(obs))
    print("Obs:        [agent_x,  agent_y,  block_x,  block_y,    block_angle]")
    print("Action: ", repr(action))
    print("Action:   [target_agent_x, target_agent_y]")

Obs:  array([240.3717, 113.6749, 292.    , 351.    ,   2.9196])
Obs:        [agent_x,  agent_y,  block_x,  block_y,    block_angle]
Action:  array([486.6163,  96.2254])
Action:   [target_agent_x, target_agent_y]


In [5]:
#@markdown ### **Dataset Demo**


dataset_path = "pusht_trained_state_data_100e.zarr"
if not os.path.isfile(dataset_path) and not os.path.exists(dataset_path):
    raise FileExistsError(f"{dataset_path} does not exist. Please collect data first.")

# parameters
pred_horizon = 16
obs_horizon = 2
action_horizon = 8
#|o|o|                             observations: 2
#| |a|a|a|a|a|a|a|a|               actions executed: 8
#|p|p|p|p|p|p|p|p|p|p|p|p|p|p|p|p| actions predicted: 16

# create dataset from file
dataset = PushTStateDataset(
    dataset_path=dataset_path,
    pred_horizon=pred_horizon,
    obs_horizon=obs_horizon,
    action_horizon=action_horizon
)
# save training data statistics (min, max) for each dim
stats = dataset.stats

# create dataloader
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=256,
    num_workers=1,
    shuffle=True,
    # accelerate cpu-gpu transfer
    pin_memory=True,
    # don't kill worker process afte each epoch
    persistent_workers=True
)

# visualize data in batch
batch = next(iter(dataloader))
print("batch['obs'].shape:", batch['obs'].shape)
print("batch['action'].shape", batch['action'].shape)

batch['obs'].shape: torch.Size([256, 2, 5])
batch['action'].shape torch.Size([256, 16, 2])


In [6]:
## Diffusion Model

# observation and action dimensions corrsponding to
# the output of PushTEnv
obs_dim = 5
action_dim = 2

# for this demo, we use DDPMScheduler with 100 diffusion iterations
num_diffusion_iters = 100

noise_pred_net, noise_scheduler, device = create_diffusion_model(obs_horizon, pred_horizon, obs_dim, action_dim, num_diffusion_iters)

number of parameters: 6.535322e+07


In [7]:
# Training takes about an hour. If you don't want to wait, skip to the next cell
# to load pre-trained weights
num_epochs = 30
ema_noise_pred_net = train_diffusion_model(noise_pred_net, noise_scheduler, dataloader, device, 
                          num_epochs, obs_horizon)
torch.save(ema_noise_pred_net.state_dict(), 'pusht_state_100ep.ckpt')

Epoch: 100%|██████████| 30/30 [06:13<00:00, 12.46s/it, loss=0.0364]


In [8]:
#@markdown ### **Loading Pretrained Checkpoint**
#@markdown Set `load_pretrained = True` to load pretrained weights.

load_pretrained = False
if load_pretrained:
  ckpt_path = "pusht_state_100ep.ckpt"
  if not os.path.isfile(ckpt_path):
      raise FileExistsError(f"{ckpt_path} does not exist. Please train first.")

  state_dict = torch.load(ckpt_path, map_location='cuda')
  ema_noise_pred_net = noise_pred_net
  ema_noise_pred_net.load_state_dict(state_dict)
  print('Pretrained weights loaded.')
else:
  print("Skipped pretrained weight loading.")

Skipped pretrained weight loading.


In [9]:
print("EMA:", type(ema_noise_pred_net))
# limit enviornment interaction to 200 steps before termination
max_steps = 5000
env = PushTEnv()
# use a seed >200 to avoid initial states seen in the training dataset
env.seed(100000)

print(type(ema_noise_pred_net))

rewards, imgs = run_inference_diffusion_model(ema_noise_pred_net, noise_scheduler, env, stats, device, 
        obs_horizon, pred_horizon, action_horizon, action_dim, max_steps, num_diffusion_iters)

# Print out the maximum target coverage
print('Score: ', max(rewards))

# visualize
from IPython.display import Video
vwrite('vis.mp4', imgs)
Video('vis.mp4', embed=True, width=256, height=256)

EMA: <class 'models.ConditionalUnet1D.ConditionalUnet1D'>
<class 'models.ConditionalUnet1D.ConditionalUnet1D'>


Eval PushTStateEnv:   7%|▋         | 362/5000 [00:31<06:38, 11.64it/s, reward=1]       

Score:  1.0
